<a href="https://colab.research.google.com/github/avinseth/Prompt-Engineering-Techniques/blob/Coding-Hub/Gemini_Guided_practice_building_llm_apps_day3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install langchain-community
!pip install langchain-google-genai
!pip install langchain-text-splitters
!pip install google-generativeai
!pip install faiss-cpu
!pip install sentence-transformers
!pip install pypdf


In [ ]:
# -*- coding: utf-8 -*-
"""
Guided_Practice_Loader_Splitter_Embeddings_Gemini.py

=====================================================
LangChain Loader, Splitter, and Embeddings (Gemini)
=====================================================
"""

import os
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
import faiss
import warnings
warnings.filterwarnings("ignore")


# =====================================================
# Step 0: Set Gemini API Key
# =====================================================

# os.environ["GOOGLE_API_KEY"] = ""


# =====================================================
# Step 1: Load text data using TextLoader
# =====================================================

text_loader = TextLoader("state_of_union.txt")
text_documents = text_loader.load()

print(text_documents[0].page_content[:100])  # First 100 characters







Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and th


In [ ]:
# =====================================================
# Step 2: Load PDF using PyPDFLoader
# =====================================================

pdf_loader = PyPDFLoader("michael_resume.pdf")
pdf_pages = pdf_loader.load_and_split()

print(pdf_pages[0].page_content[:100])  # First 100 characters of first page




CURRICULUM VITAE :  
M ichael M . Scott OBE, B.Sc., Dip.Ed  
 
Home address:  Strome House     Date 


In [ ]:
# =====================================================
# Step 3: Split documents into chunks
# =====================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1024,
    chunk_overlap=64
)

split_docs = text_splitter.split_documents(pdf_pages)

print("Number of chunks:", len(split_docs))




Number of chunks: 15


In [ ]:
# =====================================================
# Step 4: HuggingFace Embeddings
# =====================================================

MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"

hf_embeddings = HuggingFaceEmbeddings(model_name=MODEL_NAME)

sample_text = split_docs[0].page_content
hf_embedding_result = hf_embeddings.embed_documents([sample_text])

print("HF embedding length:", len(hf_embedding_result[0]))




modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

HF embedding length: 768


In [ ]:
# =====================================================
# Step 6: Create FAISS Vector Store
# =====================================================

faiss_index = FAISS.from_documents(
    split_docs,
    hf_embeddings
)





In [ ]:
# =====================================================
# Step 7: Similarity Search
# =====================================================

query = "What is the candidate's skill sets?"

results = faiss_index.similarity_search_with_score(
    query,
    k=5
)

print("\nTop 2 Similar Documents:")
for doc, score in results:
    print("Score:", score)
    print(doc.page_content[:300])
    print("-" * 50)


Top 2 Similar Documents:
Score: 1.4273027
Computer knowledge 
I am reasonably fluent in basic PC computer skills, using Windows XP, Word, WordPro, Excel, PowerPoint, 
Adobe Photoshop Elements, e-mail, internet etc.  I have full computer and broadband facilities at home. 
 
Other interests 
Botanising (especially mountain flowers), travel, w
--------------------------------------------------
Score: 1.5560662
Aberdeen College of Education (1973 - 1974): 
    Certificate of Education in secondary education (botany, zoology and general
     science), 1974 
 
Employment history 
1974 - 1976: Assistant Education Officer, Royal Zoological Society of Scotland (Edinburgh Zoo). 
1976 - 1980:  Scottish Field Offi
--------------------------------------------------
Score: 1.5590034
CV: Michael Scott  Page 3 
cruises round Scotland and to Norway, Iceland, Greenland, the Canary Islands, the Caribbean and the 
Amazon. 
 
Broadcasting experience 
Radio: Recently worked on several Nature programmes for

In [ ]:
# Save the FAISS index to a file
faiss_index.save_local("faiss_index")

In [ ]:
# Load the persisted FAISS index from the file safely
faiss_index_loaded = FAISS.load_local(
    "faiss_index",
    hf_embeddings,
    allow_dangerous_deserialization=True  # Only safe if you trust the file
)

# Perform a similarity search with the loaded FAISS index
vector_search_result = faiss_index_loaded.similarity_search_with_score(
    "What is the candidate's skill sets?",
    k=2
)
for i, (doc, score) in enumerate(vector_search_result, start=1):
    print(f"\n=== Result {i} | Score: {score:.4f} ===")
    print(doc.page_content.strip())


=== Result 1 | Score: 1.4273 ===
Computer knowledge 
I am reasonably fluent in basic PC computer skills, using Windows XP, Word, WordPro, Excel, PowerPoint, 
Adobe Photoshop Elements, e-mail, internet etc.  I have full computer and broadband facilities at home. 
 
Other interests 
Botanising (especially mountain flowers), travel, walking, Scottish islands, gardening, photography, 
computers, rugby supporter, cinema, good wine, Runrig concerts (!). 
 
[updated, 26.03.08]

=== Result 2 | Score: 1.5561 ===
Aberdeen College of Education (1973 - 1974): 
    Certificate of Education in secondary education (botany, zoology and general
     science), 1974 
 
Employment history 
1974 - 1976: Assistant Education Officer, Royal Zoological Society of Scotland (Edinburgh Zoo). 
1976 - 1980:  Scottish Field Officer, Wildlife Youth Service, World Wildlife Fund (including schools 
  lecturing and running adventure holidays and field courses for young people). 
1990 – 2004: Scottish Co-ordinator of Pl